# DiTSim 官方基线训练与生成

目标：在同一份已填补训练数据上运行官方 DiTSim，按目标 `Subclass` 配额生成所有区域，保留末尾 1000 个复制区域。正式运行使用完整细胞和完整区域轴；目标文件只提供区域轴、细胞 ID 和类型配额，目标 `X` 不参与模型。

官方来源：[BioX-NKU/DiTSim](https://github.com/BioX-NKU/DiTSim/tree/52231d81ee7d98c6222ad51539624e1204ebf6a9)，固定提交 `52231d81ee7d98c6222ad51539624e1204ebf6a9`。源码逐文件 SHA256 记录在 `baseline/DiTSim/sources.json`；`vendor/` 仅缓存必要源码、原始环境文件和许可证，不下载示例数据或权重。


## 运行约定与验证

使用 `simscDNAm` 内核。可通过 `BASELINE_TRAINING`、`BASELINE_TARGET`、`BASELINE_OUTPUT`、`BASELINE_SEED`、`BASELINE_DEVICE` 设置输入、独立输出目录、种子和设备。默认正式运行 AE 200 轮、DiT 2400 轮；DiT 以官方 `quick_start.py` 的 2400 轮为准，README 示例的 1600 轮未采用。AE 保留官方 `end_loss=0.01` 行为：源码命中阈值时保存但未跳出循环。

共享缓存 `TrainingImputed.h5ad` 只用真实训练集逐区域中位数填补；全缺失列沿用统一预处理规则。为兼容官方 loader 的 `mat.A`，稀疏输入仅转为 `numpy.ndarray float32`，不增加归一化、特征筛选、掩码、对齐、量化或投影。标签嵌入的 `num_classes` 取训练中实际类型数，其余模型与损失保持官方值。

官方 `sampling` 已内置下限零截断；随后统一 `clip(0,1)`，拒绝 NaN/Inf，不修改复制区域。最后核对完整区域轴、目标类型配额，并逐元素回读输出。采样模板缺少区域元数据，输出仅恢复训练 `var` 与列名，不改变列顺序或数值。

服务器代码位于 `/home/sccasimp/program/xb/simscDNAm`，输出限制在 `/home/sccasimp/data/xb/outputs/simscDNAm`。依赖须预先准备，本 notebook 不安装包。依赖及兼容性见 `baseline/DiTSim/README.md`。


In [ ]:
import gc
import hashlib
import importlib.metadata as package_metadata
import json
import os
from pathlib import Path
import random
import sys
import urllib.request

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
import torch

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "data" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from data.baseline import prepare_imputed_training, clip_generated_matrix, hash_text
from data.paths import OUTPUTS, check_output, require_environment


In [ ]:
def load_official():
    """下载并校验固定提交的官方源码，不改模型、训练器或采样算法。"""
    global AutoEncoder, AutoencoderTrainer, DiT, DitTrainer
    source_root = ROOT / "baseline" / "DiTSim"
    manifest = json.loads((source_root / "sources.json").read_text(encoding="utf-8"))
    vendor = source_root / "vendor"
    vendor.mkdir(parents=True, exist_ok=True)
    for relative, expected in manifest["sha256"].items():
        path = vendor / relative
        if not path.is_file():
            url = f"https://raw.githubusercontent.com/BioX-NKU/DiTSim/{manifest['commit']}/{relative}"
            payload = urllib.request.urlopen(url, timeout=60).read()
            if hashlib.sha256(payload).hexdigest() != expected:
                raise ValueError(f"官方源码下载校验失败：{relative}")
            path.write_bytes(payload)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError(f"官方源码已变化：{relative}")
    # 官方 checkpoint 含 NumPy 数组；兼容 PyTorch 2.6+ 的 torch.load 默认值变化。
    os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
    runtime = source_root / ".runtime" / "packages"
    if runtime.is_dir():
        sys.path.insert(0, str(runtime))
    sys.path.insert(0, str(vendor))
    from ae import AutoEncoder
    from ae_trainer import AutoencoderTrainer
    from transformer import DiT
    from dit_trainer import DitTrainer


def read_filled_training(filled_path):
    """只读已填补训练文件；稀疏到稠密只改变官方 loader 的输入容器。"""
    training = ad.read_h5ad(filled_path)
    values = training.X.toarray() if sparse.issparse(training.X) else training.X
    training.X = np.asarray(values, dtype=np.float32)
    for start in range(0, training.n_obs, 32):
        if not np.isfinite(training.X[start:start + 32]).all():
            raise ValueError("已填补训练矩阵必须全部为有限值")
    return training


def sample_quotas(training_obs, target_obs):
    """目标 obs 只决定生成 Subclass 配额，不传入目标表达矩阵。"""
    for name, obs in (("训练", training_obs), ("目标", target_obs)):
        if "Subclass" not in obs or obs["Subclass"].isna().any():
            raise ValueError(f"{name}缺少有效 Subclass")
    trained = set(training_obs["Subclass"].astype(str))
    labels = target_obs["Subclass"].astype(str)
    unseen = set(labels) - trained
    if unseen:
        raise ValueError(f"目标含未训练类型：{sorted(unseen)}")
    return {str(label): int(count) for label, count in labels.value_counts(sort=False).items() if count > 0}


def train_and_sample(filled_path, target_obs, run_dir, seed=42, device="cuda:0", pilot=False):
    """调用官方两阶段训练与 sampling，唯一数值后处理是 clip[0,1]。"""
    training = read_filled_training(filled_path)
    target_obs = target_obs.copy()
    if pilot:
        # 接口试跑单独记录，不作正式结果：每类至多两个细胞、前 256 列、一轮训练。
        training = training[np.flatnonzero(training.obs.groupby("Subclass", observed=True).cumcount().to_numpy() < 2),
                            :min(256, training.n_vars)].copy()
        target_obs = target_obs.loc[target_obs.groupby("Subclass", observed=True).cumcount() < 2].copy()
    quotas = sample_quotas(training.obs, target_obs)
    num_classes = int(training.obs["Subclass"].nunique())
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    for directory in ("ae_models", "dit_models", "optim"):
        (run_dir / directory).mkdir(exist_ok=True)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    ae_parameters = dict(input_dim=training.n_vars, latent_dim=256, hidden_dim=[1024, 1024],
                         embed_dim=128, reshape=True, cfg=True, num_classes=num_classes)
    dit_parameters = dict(input_size=256, patch_size=2, hidden_size=768, depth=4,
                          num_heads=8, num_classes=num_classes)
    ae_training = dict(epochs=1 if pilot else 200, end_loss=0.01, batch_size=32,
                       lr=5e-4, weight_decay=0.01, cs_loss_ratio=0.0, save_freq=10,
                       save_path=str(run_dir / "ae_models" / "autoencoder.pt"),
                       optim_path=str(run_dir / "optim" / "autoencoder.pt"))
    dit_training = dict(dit_name="DiTSim", epochs=1 if pilot else 2400, save_freq=100,
                        batch_size=32, lr=1e-4, weight_decay=0.01,
                        save_directory=str(run_dir / "dit_models"),
                        optim_path=str(run_dir / "optim" / "dit.pt"))
    sources = json.loads((ROOT / "baseline" / "DiTSim" / "sources.json").read_text(encoding="utf-8"))
    versions = {}
    for name in ("torch", "timm", "torchvision", "numpy", "pandas", "scipy", "anndata", "scanpy"):
        try:
            versions[name] = package_metadata.version(name)
        except package_metadata.PackageNotFoundError:
            versions[name] = None
    configuration = {
        "模式": "pilot" if pilot else "full", "正式结果": not pilot,
        "官方源码": sources, "训练来源": str(Path(filled_path).resolve()),
        "训练形状": list(training.shape), "生成形状": [sum(quotas.values()), training.n_vars],
        "训练细胞ID哈希": hash_text(training.obs_names), "区域轴哈希": hash_text(training.var_names),
        "目标Subclass配额": quotas, "随机种子": seed, "设备": device,
        "AE参数": ae_parameters, "AE训练": ae_training, "DiT参数": dit_parameters, "DiT训练": dit_training,
        "采样": {"sampler": "ddim", "acc_rate": 10, "batch_size": 32},
        "训练预处理": "仅训练逐区域中位数填补；numpy.ndarray float32 容器转换",
        "生成后处理": "官方 sampling 内置 clamp(min=0)；随后仅 clip(0,1)",
        "依赖": versions,
    }
    (run_dir / "config.json").write_text(json.dumps(configuration, ensure_ascii=False, indent=2), encoding="utf-8")
    print({"模式": configuration["模式"], "训练形状": training.shape, "目标配额": quotas}, flush=True)

    autoencoder = AutoEncoder(**ae_parameters)
    ae_trainer = AutoencoderTrainer(model=autoencoder, adata=training, label_name="Subclass", device=device)
    ae_trainer.train(**ae_training)
    # 官方 AE 优化器不再用于 DiT 阶段，释放它以免保留全输入维度的优化器状态。
    del ae_trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    dit = DiT(**dit_parameters)
    dit_trainer = DitTrainer(adata=training, model=dit, ae=autoencoder, label_name="Subclass",
                             device=device, device_dis=None)
    dit_trainer.load_ae(ae_training["save_path"])
    dit_trainer.train(**dit_training)
    raw_path = run_dir / "official_generated.h5ad"
    dit_trainer.sampling(quotas, batch_size=32,
                        latent_savepath=str(run_dir / "generated_latent.npy"),
                        result_savepath=str(raw_path), sampler="ddim", acc_rate=10)

    generated = ad.read_h5ad(raw_path)
    if generated.shape != (sum(quotas.values()), training.n_vars):
        raise ValueError(f"官方生成形状不符合完整区域轴：{generated.shape}")
    actual_quotas = generated.obs["Subclass"].astype(str).value_counts(sort=False).to_dict()
    if actual_quotas != quotas:
        raise ValueError("官方生成 Subclass 配额不一致")
    values, clip_report = clip_generated_matrix(generated.X)
    obs = pd.DataFrame({"Subclass": generated.obs["Subclass"].astype(str).to_numpy(), "origin": "Synthetic"},
                       index=pd.Index([f"synthetic_{i:06d}" for i in range(1, len(generated) + 1)], name="cell_id"))
    # 官方 sampling 的零模板没有原始区域元数据；按未变动的完整列顺序恢复训练 var。
    output_path = run_dir / "data.h5ad"
    partial_path = output_path.with_suffix(".h5ad.partial")
    ad.AnnData(values, obs=obs, var=training.var.copy()).write_h5ad(partial_path, compression="gzip", compression_opts=1)
    partial_path.replace(output_path)
    written = ad.read_h5ad(output_path, backed="r")
    try:
        if written.shape != values.shape or not written.var_names.equals(training.var_names):
            raise AssertionError("输出完整区域轴核对失败")
        if written.obs["Subclass"].astype(str).value_counts(sort=False).to_dict() != quotas:
            raise AssertionError("输出目标 Subclass 配额核对失败")
        for start in range(0, len(values), 32):
            if not np.array_equal(written.X[start:start + 32], values[start:start + 32]):
                raise AssertionError("输出逐元素回读核对失败")
    finally:
        written.file.close()
    verification = {"状态": "完成", "正式结果": not pilot, "形状": list(values.shape),
                    "目标Subclass配额": quotas, "实际Subclass配额": actual_quotas,
                    "完整区域轴": True, "逐元素回读核对": True, "官方源码未修改": True,
                    "生成裁剪": clip_report, "目标矩阵参与训练或生成": False}
    (run_dir / "verification.json").write_text(json.dumps(verification, ensure_ascii=False, indent=2), encoding="utf-8")
    print(verification, flush=True)
    return output_path


def run_baseline():
    """环境变量入口，默认执行完整正式训练。"""
    require_environment()
    prepared = OUTPUTS / "preprocessing" / "liu2023" / "full"
    training_path = Path(os.environ.get("BASELINE_TRAINING", str(prepared / "TrainingReal.h5ad"))).expanduser().resolve()
    target_path = Path(os.environ.get("BASELINE_TARGET", str(prepared / "HeldoutReal.h5ad"))).expanduser().resolve()
    output_root = check_output(os.environ.get("BASELINE_OUTPUT", str(OUTPUTS / "baseline" / "liu2023" / "full" / "DiTSim")))
    pilot = os.environ.get("BASELINE_PILOT", "0") == "1"
    run_dir = output_root / "pilot" if pilot else output_root
    run_dir.mkdir(parents=True, exist_ok=True)
    filled_path, target_obs, imputation_report = prepare_imputed_training(
        training_path, target_path, training_path.parent / "imputed_baselines")
    (run_dir / "imputation.json").write_text(json.dumps(imputation_report, ensure_ascii=False, indent=2), encoding="utf-8")
    load_official()
    return train_and_sample(filled_path, target_obs, run_dir,
                            seed=int(os.environ.get("BASELINE_SEED", "42")),
                            device=os.environ.get("BASELINE_DEVICE", "cuda:0"), pilot=pilot)


## 执行训练与生成

下面单元格才会开始训练。成功后输出 `data.h5ad`、配置、填补记录、验证记录、官方中间输出及检查点。

可先设置 `BASELINE_PILOT=1` 作独立接口试跑：只取每类至多两个训练/目标细胞和前 256 列，两个阶段各一轮，写入 `BASELINE_OUTPUT/pilot/` 并标记为非正式结果。默认未设置时使用全细胞、全区域和官方完整参数；正式评估只读取正式目录的 `data.h5ad`。


In [ ]:
OUTPUT_PATH = run_baseline()
OUTPUT_PATH
